In [1]:
# ============================================================
# DUAL-MODE LIVE DASHBOARD - Compact Version
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ====================== SETTINGS ======================
Z_ENTRY     = -1.5
ATR_MULT    = 2.0
SMA_WINDOW  = 20
ATR_WINDOW  = 14
TREND_SMA   = 50

MOMENTUM_BUCKET = ["NVDA", "META", "NET"]
QUALITY_BUCKET  = ["NOW", "MSFT", "GOOGL", "PANW", "CRWD", "DDOG", "CRM"]
# ======================================================


def get_data(ticker, days=120):
    end = datetime.now().date() + timedelta(days=1)
    start = end - timedelta(days=days)
    try:
        df = yf.download(ticker, start=start.strftime('%Y-%m-%d'),
                         end=end.strftime('%Y-%m-%d'),
                         multi_level_index=False, progress=False, auto_adjust=True)
        if df.empty:
            return None
        return df[['Open', 'High', 'Low', 'Close']].dropna().copy()
    except:
        return None


def get_levels(ticker, use_filter):
    df = get_data(ticker)
    if df is None or len(df) < 60:
        return None

    df = df.copy()
    df['sma'] = df['Close'].rolling(SMA_WINDOW).mean()
    df['std'] = df['Close'].rolling(SMA_WINDOW).std()
    df['zscore'] = (df['Close'] - df['sma']) / df['std']
    df['trend_sma'] = df['Close'].rolling(TREND_SMA).mean()

    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift(1)).abs(),
        (df['Low'] - df['Close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr'] = tr.rolling(ATR_WINDOW).mean()

    last = df.iloc[-1]
    close = float(last['Close'])
    sma = float(last['sma'])
    std = float(last['std'])
    atr = float(last['atr'])
    z = float(last['zscore'])
    trend_sma = float(last['trend_sma']) if pd.notna(last['trend_sma']) else np.nan

    buy_trigger = sma + (Z_ENTRY * std)
    initial_stop = buy_trigger - (ATR_MULT * atr)
    mean_exit = sma

    trend_ok = True
    if use_filter and not np.isnan(trend_sma):
        trend_ok = close > trend_sma

    signal = (z < Z_ENTRY) and trend_ok
    dist_pct = (close - buy_trigger) / close * 100

    return {
        'ticker': ticker,
        'close': close,
        'z': z,
        'buy_trigger': buy_trigger,
        'initial_stop': initial_stop,
        'mean_exit': mean_exit,
        'trend_ok': trend_ok,
        'use_filter': use_filter,
        'signal': signal,
        'dist_pct': dist_pct,
        'risk': buy_trigger - initial_stop,
        'reward': mean_exit - buy_trigger
    }


def live_dashboard():
    print("\n" + "="*95)
    print(f"DUAL-MODE LIVE DASHBOARD                    {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("="*95)

    rows = []

    for t in MOMENTUM_BUCKET:
        info = get_levels(t, use_filter=False)
        if info:
            info['bucket'] = "Momentum"
            rows.append(info)

    for t in QUALITY_BUCKET:
        info = get_levels(t, use_filter=True)
        if info:
            info['bucket'] = "Quality"
            rows.append(info)

    # ----- SUMMARY TABLE -----
    print(f"\n{'Ticker':<7} {'Bucket':<10} {'Price':>9} {'Z':>7} {'Trigger':>9} {'Stop':>9} {'Exit':>9} {'Dist%':>7} {'Signal'}")
    print("-"*95)

    for r in rows:
        sig = "← BUY" if r['signal'] else ""
        print(f"{r['ticker']:<7} {r['bucket']:<10} ${r['close']:>8.2f} {r['z']:>7.2f} "
              f"${r['buy_trigger']:>8.2f} ${r['initial_stop']:>8.2f} ${r['mean_exit']:>8.2f} "
              f"{r['dist_pct']:>6.1f}% {sig}")

    print("-"*95)

    # ----- DETAILS only for close or active signals -----
    interesting = [r for r in rows if r['signal'] or r['dist_pct'] < 12]

    if interesting:
        print("\nDETAILED LEVELS (active signal or within 12% of trigger)")
        print("="*95)
        for r in interesting:
            mode = "Quality + 50SMA filter" if r['use_filter'] else "Momentum (no filter)"
            print(f"\n{r['ticker']}   [{mode}]")
            print(f"  Price {r['close']:.2f}  |  Z {r['z']:.2f}")
            print(f"  BUY TRIGGER   : ${r['buy_trigger']:.2f}")
            print(f"  Initial Stop  : ${r['initial_stop']:.2f}")
            print(f"  Mean-Rev Exit : ${r['mean_exit']:.2f}  (Z>0)")
            print(f"  Risk/Reward   : ${r['risk']:.2f} / ${r['reward']:.2f}")
            if r['signal']:
                print(f"  >>> ALL-IN BUY SIGNAL <<<")
    else:
        print("\nNo stocks are close to a buy trigger right now (all > 12% away).")

    print("\n" + "="*95)
    print("RULES:  Momentum = Z<-1.5 only  |  Quality = Z<-1.5 AND Close>50SMA")
    print("        Both: Trail stop = Close-2×ATR  |  Also exit when Z>0")
    print("="*95)


# Run daily
live_dashboard()


DUAL-MODE LIVE DASHBOARD                    2026-07-13 17:03

Ticker  Bucket         Price       Z   Trigger      Stop      Exit   Dist% Signal
-----------------------------------------------------------------------------------------------
NVDA    Momentum   $  203.53    0.27 $  192.81 $  179.01 $  201.88    5.3% 
META    Momentum   $  656.73    1.95 $  536.76 $  482.37 $  588.99   18.3% 
NET     Momentum   $  269.53    1.42 $  214.74 $  188.93 $  242.90   20.3% 
NOW     Quality    $  111.26    1.47 $   91.97 $   80.77 $  101.72   17.3% 
MSFT    Quality    $  390.99    0.91 $  363.35 $  338.72 $  380.55    7.1% 
GOOGL   Quality    $  352.51   -0.54 $  343.31 $  323.19 $  357.62    2.6% 
PANW    Quality    $  330.30    0.63 $  271.15 $  236.15 $  312.83   17.9% 
CRWD    Quality    $  187.91    0.55 $  164.23 $  145.04 $  181.52   12.6% 
DDOG    Quality    $  260.24    0.97 $  216.68 $  190.67 $  243.14   16.7% 
CRM     Quality    $  171.22    1.67 $  150.51 $  137.98 $  160.33   12.1% 